### 1. Common imports, loading and transforming required datafiles.
**Run this first for all cases.**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch.utils.data import TensorDataset, DataLoader
import torch.nn.functional as F
from sklearn.preprocessing import MinMaxScaler

# Filepaths to pre-processed datasets
load_path = '../data/florida_load.csv'

### 2. PyTorch setup
**Run this cell before training or testing.**

In [ ]:
# Load the time-series load data
df_load = pd.read_csv(load_path)

# Extract node IDs (FIPS codes) from the column headers
county_fips = list(df_load.columns[1:])
num_nodes = len(county_fips)

# 1. Process the Time-Series Data
# Extract only the numerical load values (Rows = Time, Cols = Counties)
load_values = df_load.iloc[:, 1:].values

# Neural networks require normalized data for stable training
scaler = MinMaxScaler()
load_normalized = scaler.fit_transform(load_values)

# Transpose so rows are counties (nodes) and cols are time: Shape [num_nodes, Total_Hours]
load_normalized = load_normalized.T

# Create sliding windows (Past 24 hours -> Predict Next 1 hour)
window_size = 24
dataset = []

# To keep training fast for testing, let's use a subset of the data (e.g., first 5000 hours)
# You can remove the slicing [:5000] to train on the full 2016-2023 dataset later
# total_time_steps = min(5000, load_normalized.shape[1])

# We can extract time subseries indices for specific years.
#       start -> end
# 2016:     0 ->  8783
# 2017:  8784 -> 17543
# 2018: 17544 -> 26303
# 2019: 26304 -> 35063
# 2020: 35064 -> 43847
# 2021: 43848 -> 52607
# 2022: 52608 -> 61367
# 2023: 61368 -> 70127

# Select bounds for training and testing subseries
train_start_t = 43848
train_end_t = 61367
test_start_t = 61368
test_end_t = 70127

train_features = []
train_labels = []
test_features = []
test_labels = []

for t in range(train_start_t, train_end_t + 1 - window_size):
    # Node features: past 'window_size' hours (Shape: [num_nodes, 24])
    x = torch.tensor(load_normalized[:, t : t + window_size], dtype=torch.float)
    # Target: the load at the very next hour (Shape: [num_nodes, 1])
    y = torch.tensor(load_normalized[:, t + window_size], dtype=torch.float)
    train_features.append(x.flatten())
    train_labels.append(y)

for t in range(test_start_t, test_end_t + 1 - window_size):
    # Node features: past 'window_size' hours (Shape: [num_nodes, 24])
    x = torch.tensor(load_normalized[:, t : t + window_size], dtype=torch.float)
    # Target: the load at the very next hour (Shape: [num_nodes, 1] scalar)
    y = torch.tensor(load_normalized[:, t + window_size], dtype=torch.float)
    test_features.append(x.flatten())
    test_labels.append(y)

train_dataset = TensorDataset(torch.stack(train_features), torch.stack(train_labels))
test_dataset = TensorDataset(torch.stack(test_features), torch.stack(test_labels))
print(f"Created {len(train_dataset)} tensors for training.")
print(f"Created {len(test_dataset)} tensors for testing.")

#for t in range(total_time_steps - window_size):
#    # Node features: past 'window_size' hours (Shape: [num_counties, 24])
#    x = torch.tensor(load_normalized[:, t : t + window_size], dtype=torch.float)
#
#    # Target: the load at the very next hour (Shape: [num_counties, 1])
#    y = torch.tensor(load_normalized[:, t + window_size], dtype=torch.float).unsqueeze(1)
#
#    # Create a PyG Data object for this specific time snapshot
#    graph_snapshot = Data(x=x, edge_index=edge_index, edge_weight=edge_attr, y=y)
#    dataset.append(graph_snapshot)

# print(f"Created {len(dataset)} graph snapshots for training/testing.")

# Split into Train (80%) and Test (20%)
# train_size = int(len(dataset) * 0.8)
# train_dataset = dataset[:train_size]
# test_dataset = dataset[train_size:]

# DataLoaders batch multiple graph snapshots together for faster training
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

### 3. Model definition and implementation
**Run this cell before training or testing.**

In [ ]:
class LoadForecastingFF(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        # Layer 1
        self.fc1 = torch.nn.Linear(in_channels, hidden_channels)
        # Layer 2
        self.fc2 = torch.nn.Linear(hidden_channels, out_channels)

        # Initialize weights (He initialization)
        torch.nn.init.kaiming_uniform_(self.fc1.weight)
        torch.nn.init.kaiming_uniform_(self.fc2.weight)

    def forward(self, data):
        x = data
        x = self.fc1(x)
        x = F.relu(x)
        x = self.fc2(x)
        return x

# Initialize the model
# in_channels = 24*num_counties (past 24 hours), out_channels = num_counties (next hour)
num_counties = num_nodes
model = LoadForecastingFF(in_channels=window_size*num_counties, hidden_channels=32, out_channels=num_counties)
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)
criterion = torch.nn.MSELoss()

print(model)

### 4. Training the model
You may skip training if you have an existing checkpoint.

In [ ]:
import copy
import matplotlib.pyplot as plt

# Use CUDA GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

def train():
    model.train()
    total_loss = 0
    for x, y in train_loader:
        x = x.to(device)
        y = y.to(device)
        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(train_dataset)

def validate():
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for x, y in test_loader:
            x = x.to(device)
            y = y.to(device)
            out = model(x)
            loss = criterion(out, y)
            total_loss += loss.item()
    return total_loss / len(test_dataset)

# Tracking variables
train_losses = []
test_losses = []
best_loss = float('inf')
best_model_state = None

epochs = 500 # Increased epochs for better learning

print("Starting Training...")
for epoch in range(epochs):
    train_loss = train()
    test_loss = validate()

    train_losses.append(train_loss)
    test_losses.append(test_loss)

    # Save the model if it's the best we've seen so far
    if test_loss < best_loss:
        best_loss = test_loss
        best_model_state = copy.deepcopy(model.state_dict())
        torch.save(best_model_state, 'best_load_forecasting_ff.pth')

    # Print statistics every few epochs
    if epoch == 0 or epoch % 10 == 9:
        print(f'Epoch: {epoch+1:03d}, Train Loss: {train_loss:.4f}, Test Loss: {test_loss:.4f}')

print("Training Complete! Best model saved as 'best_load_forecasting_ff.pth'")

# --- Plot the Learning Curve ---
plt.figure(figsize=(10, 5))
plt.plot(range(1, epochs + 1), train_losses, label='Train Loss (MSE)')
plt.plot(range(1, epochs + 1), test_losses, label='Test Loss (MSE)')
plt.xlabel('Epochs')
plt.ylabel('Mean Squared Error')
plt.title('GCN Training and Validation Loss')
plt.legend()
plt.grid(True)
plt.show()

### 5. Running prediction and inference
Testing and MAPE/RMSE evaluation.

In [ ]:
from torch_geometric.nn import global_add_pool

# Load the best model weights
model.load_state_dict(torch.load('best_load_forecasting_ff.pth'))
model = model.to("cpu")
model.eval()

all_true_aggregated = []
all_pred_aggregated = []

with torch.no_grad():
    for x, y in test_loader:
        # Get predictions (Normalized)
        out = model(x)        

        # In PyTorch Geometric, 'batch.batch' tells us which nodes belong to which time step
        # We use global_add_pool to sum the load across all N counties for each time step
        # This matches the inner sum: Σ y_{i,t}
        true_sum_norm = y.sum()
        pred_sum_norm = out.sum()

        # Convert tensors to numpy arrays
        true_sum_norm = true_sum_norm.cpu().numpy()
        pred_sum_norm = pred_sum_norm.cpu().numpy()

        # Inverse transform back to raw Megawatts (MW)
        # Note: Since we summed across all N counties, we need to scale it carefully.
        # An easier trick is to inverse transform the individual nodes FIRST, then sum.
        y_raw = scaler.inverse_transform(y.cpu().numpy().reshape(-1, num_nodes))
        out_raw = scaler.inverse_transform(out.cpu().numpy().reshape(-1, num_nodes))

        # Sum raw MW values across all counties (axis=1)
        true_sum_mw = np.sum(y_raw, axis=1)
        pred_sum_mw = np.sum(out_raw, axis=1)

        all_true_aggregated.extend(true_sum_mw)
        all_pred_aggregated.extend(pred_sum_mw)

# Convert lists to numpy arrays for final metric calculations
Y_true = np.array(all_true_aggregated)
Y_pred = np.array(all_pred_aggregated)

# 1. Calculate MAPE
# formula: (1/T) * Σ | (true - pred) / true |
mape = np.mean(np.abs((Y_true - Y_pred) / Y_true)) * 100

# 2. Calculate RMSE
# formula: sqrt( (1/T) * Σ (true - pred)^2 )
rmse = np.sqrt(np.mean((Y_true - Y_pred)**2))



In [ ]:
print("--- Final Model Evaluation (Aggregated Florida Load) ---")
print(f"MAPE: {mape:.2f}%")
print(f"RMSE: {rmse:.2f} MW")

# Optional: Plot a small window of predictions vs actuals
plt.figure(figsize=(12, 5))
plt.plot(Y_true[:100], label='Actual Aggregated Load (MW)', color='blue')
plt.plot(Y_pred[:100], label='Predicted Aggregated Load (MW)', color='red', linestyle='--')
plt.title('GCN Forecast vs Actual (First 100 Hours of Test Set)')
plt.xlabel('Time Steps (Hours)')
plt.ylabel('Total Load (MW)')
plt.legend()
plt.grid(True)
plt.show()

print("--- Mean and variance of ground-truth and predicted values ---")
print("Mean Y_true:", Y_true.mean())
print("Var  Y_true:", Y_true.var())
print("Mean Y_pred:", Y_pred.mean())
print("Var  Y_pred:", Y_pred.var())